# 🧠 Modeling Phase

### **Overview**
In this phase, we train and evaluate machine learning models to predict maintenance requirements.

---

### **Steps**

1️⃣ **Select Target & Features**  
- Define the target variable (`maintenance_required`).  
- Select relevant input features for prediction.

2️⃣ **Encode Categorical Variables**  
- Convert categorical columns (e.g., `machine_id`) to numeric using one-hot encoding.

3️⃣ **Feature Scaling**  
- Standardize numerical features to improve model performance.

4️⃣ **Train/Test Split**  
- Split data into training (80%) and testing (20%).  
- Keep class distribution balanced using stratification.

5️⃣ **Train Models**  
- Train **Random Forest Classifier**.  
- Train **XGBoost Classifier**.

6️⃣ **Hyperparameter Tuning**  
- Use **GridSearchCV** or **RandomizedSearchCV** to find the best parameters for each model.

7️⃣ **Evaluate Models**  
- Compute metrics:  
  - Accuracy  
  - F1-score (macro/weighted)  
  - ROC-AUC  
  - Confusion Matrix  
  - Classification Report

8️⃣ **Select Best Model**  
- Compare tuned models.  
- Choose the one with the highest F1-score / ROC-AUC.

9️⃣ **Save Model**  
- Save the best-performing model as a **Pickle file** for deployment.  
- Example filename: `best_model_machine_fail.pkl`

---

**Note:** Each step can be implemented in separate code cells for clarity and organization.


## 📌 Loading the Preprocessed Dataset

After completing all preprocessing steps (handling missing values, outlier removal, feature scaling, and PCA) in the data preprocessing notebook, we saved the final cleaned dataset as:

``preprocessed_smart_data.csv``

In this modeling notebook, we simply load the processed file instead of repeating the entire preprocessing pipeline:




In [50]:
import pandas as pd

hd_clean = pd.read_csv("preprocessed_smart_data.csv")
print("Loaded processed dataset:", hd_clean.shape)


Loaded processed dataset: (98538, 32)


## 1️⃣ Select Target & Features  
Define the target variable (`maintenance_required`) and select the important features used for prediction.  



In [45]:
y = hd_clean['maintenance_required']

X = hd_clean[['temperature', 'vibration', 'humidity', 'pressure',
              'energy_consumption', 'machine_status',
              'anomaly_flag', 'downtime_risk']]


## 2️⃣ Encode Categorical Variables  
 Convert categorical columns such as `machine_id` to numeric form using one-hot encoding.  



In [46]:
X = pd.get_dummies(X.join(hd_clean[['machine_id']]), drop_first=True)


## 3️⃣ Feature Scaling  
 Standardize numerical data to improve model stability and performance.  



In [47]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## 4️⃣ Train/Test Split  
 Split the dataset into training (80%) and testing (20%) while keeping class balance using stratification.  



In [48]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


## 5️⃣ Train Random Forest Model  
 Train a Random Forest classifier with 300 trees and balance class weights due to the scarcity of failure cases.  



In [51]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight='balanced'
)

rf.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

## 6️⃣ Make Predictions  
 Use the trained model to predict labels for the test dataset.  



In [52]:
y_pred = rf.predict(X_test)


## 7️⃣ Evaluate Model Performance  
 Assess model accuracy, F1-score, confusion matrix, and generate a classification report.  



In [54]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("===== Random Forest Report =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1-score (macro):", f1_score(y_test, y_pred, average='macro'))
print("F1-score (weighted):", f1_score(y_test, y_pred, average='weighted'))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Failure', 'Failure']))


===== Random Forest Report =====
Accuracy: 0.9834077531966714
F1-score (macro): 0.9722911828501143
F1-score (weighted): 0.9831165476768986

Confusion Matrix:
 [[15932     0]
 [  327  3449]]

Classification Report:
              precision    recall  f1-score   support

  No Failure       0.98      1.00      0.99     15932
     Failure       1.00      0.91      0.95      3776

    accuracy                           0.98     19708
   macro avg       0.99      0.96      0.97     19708
weighted avg       0.98      0.98      0.98     19708



## 1️⃣ Train XGBoost Classifier  
Initialize and train the XGBoost classifier with 300 trees.  



In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

## 2️⃣ Make Predictions  
 Predict the class labels and the probability of failure.  


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


## 3️⃣ Evaluate XGBoost Model  
 Compute accuracy, F1-score, ROC-AUC, confusion matrix, and classification report.  



In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report

acc = accuracy_score(y_test, y_pred)
f1  = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
cm  = confusion_matrix(y_test, y_pred)

print("Accuracy:", acc)
print("F1-score:", f1)
print("ROC-AUC:", roc)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.9833570123807591
F1-score: 0.9546083587046775
ROC-AUC: 0.958238845203343
Confusion Matrix:
 [[15931     1]
 [  327  3449]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99     15932
           1       1.00      0.91      0.95      3776

    accuracy                           0.98     19708
   macro avg       0.99      0.96      0.97     19708
weighted avg       0.98      0.98      0.98     19708



## 1️⃣ Random Forest — Hyperparameter Tuning  
 Use RandomizedSearchCV to find the best hyperparameters for Random Forest.  



In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
import numpy as np

rf_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}

rf_model = RandomForestClassifier(random_state=42)

rf_random_search = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=rf_params,
    n_iter=20,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

rf_random_search.fit(X_train, y_train)

best_rf = rf_random_search.best_estimator_

print("Best Random Forest Parameters:", rf_random_search.best_params_)


Best Random Forest Parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 15, 'class_weight': 'balanced'}


## 2️⃣ XGBoost — Hyperparameter Tuning  
Tune key hyperparameters for XGBoost using RandomizedSearchCV.  



In [ ]:
from xgboost import XGBClassifier

xgb_params = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

xgb_model = XGBClassifier(
    eval_metric='logloss',
    random_state=42
)

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=xgb_params,
    n_iter=20,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

xgb_random_search.fit(X_train, y_train)

best_xgb = xgb_random_search.best_estimator_

print("Best XGBoost Parameters:", xgb_random_search.best_params_)


Best XGBoost Parameters: {'subsample': 0.6, 'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.01, 'colsample_bytree': 0.8}


## 3️⃣ Compare Best-Tuned Models  
 Evaluate both tuned models and decide which performs better.  



In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def evaluate(model, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\n===== {name} Results =====")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1-score:", f1_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    return f1_score(y_test, y_pred)

rf_f1  = evaluate(best_rf, "Random Forest")
xgb_f1 = evaluate(best_xgb, "XGBoost")

best_model = best_xgb if xgb_f1 > rf_f1 else best_rf
print("\n🔥 Best Model Selected:", "XGBoost" if best_model == best_xgb else "Random Forest")



===== Random Forest Results =====
Accuracy: 0.9834077531966714
F1-score: 0.9547404844290658
ROC-AUC: 0.9566775220800692

===== XGBoost Results =====
Accuracy: 0.9834077531966714
F1-score: 0.9547404844290658
ROC-AUC: 0.9574443536779194

🔥 Best Model Selected: Random Forest


## 4️⃣ Save Best Model using Pickle  
 Save the best-performing model to a `.pkl` file for deployment.  



In [ ]:
import pickle

with open("best_model_machine_fail.pkl", "wb") as f:
    pickle.dump(best_rf, f)

print("Best model (Random Forest) saved as best_model_machine_fail.pkl")


Best model (Random Forest) saved as best_model_machine_fail.pkl
